[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-04-memorize-or-learn-and-tune.ipynb)

# Advanced Discussion 4 — does it learn, or does it memorize? Debug, then tune

Reproduces the chapter 4 advanced discussion: the shuffled-label experiment, the overfit-one-batch and gradient-check debugging tools, an Optuna search with pruning, and L-BFGS on a badly scaled loss.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: all data comes from scikit-learn's bundled digits set.

In [ ]:
%pip install -q optuna

In [ ]:
import torch, torch.nn as nn, numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

def run(shuffle_labels, epochs=400, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    X, y = load_digits(return_X_y=True)
    X = (X / 16.0).astype("float32")
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    if shuffle_labels:
        ytr = np.random.permutation(ytr)          # destroy the X -> y relationship
    Xtr, Xte = torch.tensor(Xtr), torch.tensor(Xte)
    ytr, yte = torch.tensor(ytr), torch.tensor(yte)
    model = nn.Sequential(nn.Linear(64, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    lossf = nn.CrossEntropyLoss()
    hist = []
    for ep in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        loss = lossf(model(Xtr), ytr); loss.backward(); opt.step()
        if ep in (10, 50, 100, 200, 400):
            model.eval()
            with torch.no_grad():
                tr = (model(Xtr).argmax(1) == ytr).float().mean().item()
                te = (model(Xte).argmax(1) == yte).float().mean().item()
            hist.append((ep, round(loss.item(), 3), round(tr, 3), round(te, 3)))
    return hist

for flag in (False, True):
    print("random labels" if flag else "true labels")
    for h in run(flag): print("  epoch %d loss %.3f train_acc %.3f test_acc %.3f" % h)

In [ ]:
import torch, torch.nn as nn, numpy as np, optuna, warnings
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
optuna.logging.set_verbosity(optuna.logging.WARNING)
torch.manual_seed(0)

## 1. gradient check

In [ ]:
def grad_check(model, loss_fn, x, y, eps=1e-4, n_probe=6):
    model.double(); x = x.double()
    model.zero_grad(); loss_fn(model(x), y).backward()
    worst = 0.0
    params = [p for p in model.parameters() if p.requires_grad]
    for p in params:
        flat, gflat = p.data.view(-1), p.grad.view(-1)
        for i in torch.randperm(flat.numel())[:n_probe]:
            old = flat[i].item()
            flat[i] = old + eps; up = loss_fn(model(x), y).item()
            flat[i] = old - eps; dn = loss_fn(model(x), y).item()
            flat[i] = old
            num, ana = (up - dn) / (2 * eps), gflat[i].item()
            rel = abs(num - ana) / max(abs(num) + abs(ana), 1e-12)
            worst = max(worst, rel)
    return worst

net = nn.Sequential(nn.Linear(5, 8), nn.Tanh(), nn.Linear(8, 3))
x, y = torch.randn(4, 5), torch.tensor([0, 2, 1, 1])
print("worst relative error (correct backward):", "%.2e" % grad_check(net, nn.CrossEntropyLoss(), x, y))

class BrokenLinear(nn.Module):           # backward deliberately wrong: gradient of x is scaled by 2
    def __init__(self):
        super().__init__(); self.l = nn.Linear(5, 8)
    def forward(self, x):
        class F(torch.autograd.Function):
            @staticmethod
            def forward(ctx, a): return a * 1.0
            @staticmethod
            def backward(ctx, g): return g * 2.0
        return F.apply(self.l(x))
net2 = nn.Sequential(BrokenLinear(), nn.Tanh(), nn.Linear(8, 3))
print("worst relative error (buggy backward):", "%.2e" % grad_check(net2, nn.CrossEntropyLoss(), x, y))

## 2. overfit one batch

In [ ]:
X, Y = load_digits(return_X_y=True); X = torch.tensor(X / 16.0, dtype=torch.float32); Y = torch.tensor(Y)
m = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))
opt = torch.optim.Adam(m.parameters(), lr=1e-2); lf = nn.CrossEntropyLoss()
xb, yb = X[:32], Y[:32]
for step in range(1, 101):
    opt.zero_grad(); l = lf(m(xb), yb); l.backward(); opt.step()
    if step in (1, 25, 50, 100): print("one-batch step", step, "loss %.4f" % l.item())

## 3. Optuna with pruning

In [ ]:
Xtr, Xva, ytr, yva = train_test_split(X.numpy(), Y.numpy(), test_size=0.3, random_state=0, stratify=Y.numpy())
Xtr, Xva, ytr, yva = map(torch.tensor, (Xtr, Xva, ytr, yva))
def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    wd = trial.suggest_float("weight_decay", 1e-8, 1e-2, log=True)
    hidden = trial.suggest_categorical("hidden", [32, 64, 128, 256])
    drop = trial.suggest_float("dropout", 0.0, 0.5)
    torch.manual_seed(trial.number)
    model = nn.Sequential(nn.Linear(64, hidden), nn.ReLU(), nn.Dropout(drop), nn.Linear(hidden, 10))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    lf = nn.CrossEntropyLoss()
    for epoch in range(30):
        model.train(); opt.zero_grad(); lf(model(Xtr), ytr).backward(); opt.step()
        model.eval()
        with torch.no_grad(): acc = (model(Xva).argmax(1) == yva).float().mean().item()
        trial.report(acc, epoch)
        if trial.should_prune(): raise optuna.TrialPruned()
    return acc
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))
study.optimize(objective, n_trials=40)
done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pr = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print("optuna: complete", len(done), "pruned", len(pr), "best %.3f" % study.best_value, {k: (round(v, 5) if isinstance(v, float) else v) for k, v in study.best_params.items()})

## 4. L-BFGS on a badly scaled quadratic

In [ ]:
A = torch.diag(torch.tensor([0.2, 4.0])); w = torch.tensor([-5.0, -2.0], requires_grad=True)
f = lambda w: 0.5 * w @ A @ w
opt = torch.optim.LBFGS([w], lr=1.0, max_iter=20, line_search_fn="strong_wolfe")
def closure():
    opt.zero_grad(); l = f(w); l.backward(); return l
opt.step(closure); print("L-BFGS one step (up to 20 inner iters): w =", [round(v, 6) for v in w.tolist()], "f = %.2e" % f(w).item())
w2 = torch.tensor([-5.0, -2.0], requires_grad=True); o2 = torch.optim.SGD([w2], lr=0.4)
for _ in range(20):
    o2.zero_grad(); f(w2).backward(); o2.step()
print("SGD 20 steps: w =", [round(v, 4) for v in w2.tolist()], "f = %.2e" % f(w2).item())

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.